In [1]:
import newkernelo as newker
import kernelo as oldker
import numpy as np
import time
import logging
logging.getLogger().setLevel(logging.INFO)

## Global parameters

In [2]:
gamma_type = 'Full'
sigma_type = 'Diag'
seed = 12345678

# initialisation parameters
gllim_em_iteration = 50
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 1
gmm_em_iteration = 1
gmm_floor = 1e-12
nb_experiences = 10

# training parameters
train_max_iteration = 300
train_ratio_ll = 1e-5
train_floor = 1e-12

##### Generate random dataset

In [3]:
L, D, K, N = 4, 9, 5, 1000

x_gen = np.random.rand(N,L)
y_gen = np.random.rand(N,D)

#### Generate sobol Test Model dataset

In [4]:
L, D, K, N = 4, 9, 5, 100

# Create "physical" model
# dir_path = os.path.dirname(os.path.realpath(__file__))
# print(dir_path+"/pytest/models")
# physical_model = oldker.ExternalModelConfig("RPVModel", "rpv_model", dir_path + "/pytest/models").create()
physical_model = oldker.TestModelConfig().create()

# Create StatModel
covariances = np.random.uniform(0, 0.0001, physical_model.get_D_dimension())
stat_model = oldker.GaussianStatModelConfig("sobol", physical_model, covariances, 12345).create()

# Initialize and train GLLIM model
print("Generating dataset")
x_gen, y_gen = stat_model.gen_data(N)
print(x_gen.shape)
print(y_gen.shape)

Generating dataset
(100, 4)
(100, 9)


#### Generate sobol RPV model dataset

In [5]:
L, D, K, N = 3, 71, 10, 100

# Create "physical" model
# dir_path = os.path.dirname(os.path.realpath(__file__))
# print(dir_path+"/pytest/models")
physical_model = oldker.ExternalModelConfig("RPVModel", "rpv_model", "../../pytest/models").create()

# Create StatModel
covariances = np.random.uniform(0, 0.0001, physical_model.get_D_dimension())
stat_model = oldker.GaussianStatModelConfig("sobol", physical_model, covariances, 12345).create()

# Initialize and train GLLIM model
print("Generating dataset")
x_gen, y_gen = stat_model.gen_data(N)
print(x_gen.shape)
print(y_gen.shape)

Generating dataset
(100, 3)
(100, 71)


## OLD GLLiM

In [6]:
# Create GLLIM model, including its initialization and training configuration
learningConfig = oldker.EMLearningConfig(train_max_iteration, train_ratio_ll, train_floor)
# learningConfig=oldker.GMMLearningConfig(train_max_iteration, train_ratio_ll, train_floor)
# initConfig = oldker.MultInitConfig(seed=123456789, nb_iter_EM=10, nb_experiences=10, gmmLearningConfig=oldker.GMMLearningConfig(15, 10, 1e-12))
initConfig = oldker.MultInitConfig(seed=seed, nb_iter_EM=gllim_em_iteration, nb_experiences=nb_experiences, gmmLearningConfig=oldker.GMMLearningConfig(gmm_kmeans_iteration, gmm_em_iteration, gmm_floor))
gllim_old= oldker.GLLiM(D, L, K, gamma_type, sigma_type, initConfig, learningConfig)


In [7]:

print("Initializing GLLIM model")
gllim_old.initialize(x_gen, y_gen)
gllim_old_params_0 = gllim_old.exportModel() # you can export your gllim model parameters

print("Training model")
gllim_old.train(x_gen, y_gen)
gllim_old_params = gllim_old.exportModel() # you can export your gllim model parameters


INFO:root:Start Multi initialization
INFO:root:Initialisation : 1
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model


Initializing GLLIM model


INFO:root:	Current log likelihood : 108.208073, Best log likelihood : 108.208073
INFO:root:Initialisation : 2
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 102.303744, Best log likelihood : 108.208073
INFO:root:Initialisation : 3
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 121.207397, Best log likelihood : 121.207397
INFO:root:Initialisation : 4
INFO:root:	Generate GMM means
INFO:root:	Generate GMM covariance matrices
INFO:root:	Train the GMM model
INFO:root:	Compute Initial theta vector of the GLLiM model
INFO:root:	Train the initial GLLiM model
INFO:root:	Current log likelihood : 107.925922, Best lo

Training model


INFO:root:Iteration : 33, log likelihood : 124.943147
INFO:root:Iteration : 34, log likelihood : 124.943147
INFO:root:Iteration : 35, log likelihood : 124.943147
INFO:root:Iteration : 36, log likelihood : 124.943147
INFO:root:Iteration : 37, log likelihood : 124.943147
INFO:root:Iteration : 38, log likelihood : 124.943147
INFO:root:Iteration : 39, log likelihood : 124.943147
INFO:root:Iteration : 40, log likelihood : 124.943147
INFO:root:Iteration : 41, log likelihood : 124.943147
INFO:root:Iteration : 42, log likelihood : 124.943147
INFO:root:Iteration : 43, log likelihood : 124.943147
INFO:root:Iteration : 44, log likelihood : 124.943147
INFO:root:Iteration : 45, log likelihood : 124.943147
INFO:root:Iteration : 46, log likelihood : 124.943147
INFO:root:Iteration : 47, log likelihood : 124.943147
INFO:root:Iteration : 48, log likelihood : 124.943147
INFO:root:Iteration : 49, log likelihood : 124.943147
INFO:root:Iteration : 50, log likelihood : 124.943147
INFO:root:Iteration : 51, lo

### Get theta_0 from oldker

In [8]:
# theta_0 = newker.GLLiMParameters(L,D,K, "full", "diag")

# print(np.array(gllim_parameters_0.Pi).shape)
# print(np.array(gllim_parameters_0.A).shape)
# print(np.array(gllim_parameters_0.C).shape)
# print(np.array(gllim_parameters_0.Gamma).shape)
# print(np.array(gllim_parameters_0.B).shape)
# print(np.array(gllim_parameters_0.Sigma).shape)

# print(np.array(gllim_parameters_0.Pi))
# print(np.transpose(np.array(gllim_parameters_0.A), (1,2,0)).shape)
# print(np.array(gllim_parameters_0.C).shape)
# print(np.transpose(np.array(gllim_parameters_0.Gamma), (1,2,0)).shape)
# print(np.array(gllim_parameters_0.B).shape)
# print(np.transpose(np.array(gllim_parameters_0.Sigma), (1,2,0)).shape)

# theta_0.Pi = np.copy(np.array(gllim_parameters_0.Pi))
# theta_0.A = np.copy(np.transpose(np.array(gllim_parameters_0.A), (1,2,0)))
# theta_0.C = np.copy(np.array(gllim_parameters_0.C))
# # theta_0.Gamma = np.copy(np.transpose(np.array(gllim_parameters_0.Gamma), (1,2,0)))
# theta_0.Gamma = np.copy(gllim_parameters_0.Gamma)
# theta_0.B = np.copy(np.array(gllim_parameters_0.B))
# # theta_0.Sigma = np.copy(np.transpose(np.array(gllim_parameters_0.Sigma), (1,2,0)))
# Sigma_diag = np.zeros((K,D))
# for k in range(K):
#     Sigma_diag[k,:] = np.diag(gllim_parameters_0.Sigma[k])
# theta_0.Sigma = Sigma_diag

# print(np.sum(theta_0.Pi))
# gllim.setParams(theta_0)

## NEW GLLiM

In [9]:
new_gllim = newker.GLLiM(L,D,K,gamma_type.lower(), sigma_type.lower())

verbose = 0
X_gen = x_gen.T
Y_gen = y_gen.T

new_gllim.initialize(X_gen, Y_gen, gllim_em_iteration, gllim_em_floor, gmm_kmeans_iteration, gmm_em_iteration, gmm_floor, nb_experiences, seed, verbose)
new_gllim_params_0 = new_gllim.getParams()

new_gllim.train(X_gen, Y_gen, train_max_iteration, train_ratio_ll, train_floor, verbose)
new_gllim_params = new_gllim.getParams()


INFO: GLLiM Parameters initialized
INFO: GLLiM dimensions are (L=3, D=71, K=10)
INFO: GLLiM constraints are :
	gamma_type = 'full',
	sigma_type = 'diag'.


chol(): given matrix is not symmetric


In [10]:
print(gllim_old_params_0.Pi)
print(new_gllim_params_0.Pi)

print("\n")
print(gllim_old_params.Pi)
print(new_gllim_params.Pi)

print("\n")
print(gllim_old_params.B)
print(new_gllim_params.B)

[0.07 0.12 0.17 0.04 0.08 0.1  0.11 0.21 0.07 0.03]
[[0.11       0.12       0.08       0.08       0.09       0.1
  0.1        0.20000011 0.07999989 0.04      ]]


[0.07 0.12 0.17 0.04 0.08 0.1  0.11 0.21 0.07 0.03]
[[0.11       0.12       0.08       0.08       0.09       0.1
  0.1        0.20000011 0.07999989 0.04      ]]


[[ 5.53456658e-01  5.61316665e-01  6.32198990e-01  7.60398083e-01
   9.51430881e-01  1.20072255e+00  1.46498605e+00  1.46497844e+00
   1.20080072e+00  9.51257014e-01]
 [ 7.60445917e-01  6.32349552e-01  5.61550526e-01  5.53156113e-01
   1.46112223e-01  2.18431476e-01  2.85264724e-01  3.66781961e-01
   4.79074866e-01  6.41725184e-01]
 [ 8.77589506e-01  1.20083961e+00  1.58786543e+00  1.87369076e+00
   1.67113479e+00  1.46498622e+00  1.33918038e+00  1.36969873e+00
   3.82750271e+00  3.57478989e+00]
 [ 3.54326394e+00  2.43280909e+00  1.67118765e+00  1.12416909e+00
   7.60462305e-01  5.24294560e-01  3.67028728e-01  2.53439094e-01
   1.60053559e-01  6.45923271e-02]
 [-6.5

In [11]:
# Y_gen = np.array(y_gen[:5].T)
# pred = new_gllim.inverseDensities(Y_gen)
# print(pred.meanPredResult.mean)
# print(x_gen[:5])
# print("\n")


predicator = oldker.PredictionConfig(2, 2, 1e-10, gllim_old).create()
for i in range(5):
    print(x_gen[i])

    prediction = predicator.predict(y_gen[i], np.zeros(D))
    print(prediction.meansPred.mean)

    pred = new_gllim.inverseDensities(y_gen[i])
    print(pred.meanPredResult.mean)
    print("\n")


[0.5 0.5 0.5]
[0.62567902 0.57051817 0.49222191]
[[0.46437563 0.29515942 0.4554165 ]]


[0.75 0.25 0.25]
[0.75265298 0.20448591 0.23439512]
INFO: Inverse theta
INFO: Construct the GMM of the inverse conditional model
INFO: Compute the weighted mean of the means in the mixture
INFO: Compute the weighted covariance of the covariances in the mixture
[[0.75530298 0.21619597 0.23706966]]


[0.25 0.75 0.75]
INFO: Inverse theta
INFO: Construct the GMM of the inverse conditional model
INFO: Compute the weighted mean of the means in the mixture
INFO: Compute the weighted covariance of the covariances in the mixture
[0.84891239 0.75381467 0.89957943]
INFO: Inverse theta
INFO: Construct the GMM of the inverse conditional model
INFO: Compute the weighted mean of the means in the mixture
INFO: Compute the weighted covariance of the covariances in the mixture
[[0.19432889 0.77135772 0.62275724]]


[0.375 0.375 0.625]
[0.36472763 0.3587117  0.62201611]
INFO: Inverse theta
INFO: Construct the GMM of t

In [12]:
pred.centerPredResult.means

array([], shape=(0, 0), dtype=float64)

In [13]:
insights = new_gllim.getInsights()

In [14]:
print(insights.time)

0:00:04
